# Stage 1 — Build Full Dataset
**[STUDENT VERSION — fill in the blanks]**

- Input: `stage0_tfidf_scores.csv` — 8 semantic scores từ Stage 0
- Input: `stage1_factual.csv` — budget, family, access, crowd, best_months
- Input: `stage1_coordinates.csv` — lat, lng
- Output: `stage1_dataset.csv` và `stage1_dataset.json`

> 💡 Cells marked `# TODO` require you to fill in the code.


In [ ]:
import csv, json
from pathlib import Path
import pandas as pd


In [ ]:
BASE_DIR     = Path('.')
TFIDF_CSV    = BASE_DIR.parent / 'stage0' / 'stage0_tfidf_scores.csv'
FACTUAL_CSV  = BASE_DIR / 'stage1_factual.csv'
COORDS_CSV   = BASE_DIR / 'stage1_coordinates.csv'
DATASET_CSV  = BASE_DIR / 'stage1_dataset.csv'
DATASET_JSON = BASE_DIR / 'stage1_dataset.json'

def read_csv(path):
    with path.open('r', encoding='utf-8-sig', newline='') as f:
        return list(csv.DictReader(f))

tfidf_rows   = read_csv(TFIDF_CSV)
factual_rows = read_csv(FACTUAL_CSV)
coord_rows   = read_csv(COORDS_CSV)

print(f'TF-IDF rows: {len(tfidf_rows)}')
print(f'Factual rows: {len(factual_rows)}')
print(f'Coord rows: {len(coord_rows)}')


TF-IDF rows: 38
Factual rows: 38
Coord rows: 38


## 🔧 TODO 1 — Index by place name

Convert 3 list of rows thành 3 dict, key là `place`.  
Mục đích: tra cứu nhanh theo tên điểm thay vì loop.

```python
# Ví dụ kết quả mong muốn:
tfidf['Phong Nha'] → {'place': 'Phong Nha', 'beach': '0.072', ...}
```


In [ ]:
# TODO 1: Index 3 data sources by place name
tfidf   = None  # ← {r['place']: r for r in tfidf_rows}
factual = None  # ← tương tự
coords  = None  # ← tương tự

# Kiểm tra
print(f'Sample tfidf keys: {list(tfidf.keys())[:3]}')


## 🔧 TODO 2 — Validation

Kiểm tra: tất cả điểm trong `tfidf` có mặt trong `factual` và `coords` không?  
Nếu thiếu → raise ValueError trước khi merge.


In [ ]:
# TODO 2: Validation — tìm điểm bị thiếu
missing_factual = None  # ← [p for p in tfidf if p not in factual]
missing_coords  = None  # ← tương tự với coords

if missing_factual:
    raise ValueError(f'Missing factual data for: {missing_factual}')
if missing_coords:
    raise ValueError(f'Missing coordinates for: {missing_coords}')

print('Validation passed.')


## 🔧 TODO 3 — Merge Dataset

Với mỗi điểm đến, tạo 1 dict gồm:
- `place`, `province`, `lat`, `lng`
- 8 semantic features từ `tfidf` (convert sang float)
- 4 factual features từ `factual` (convert sang float)
- `best_months` từ `factual` — **giữ nguyên dạng string, KHÔNG convert sang float**

> ⚠️ `best_months` là string vì giá trị của nó phụ thuộc vào tháng user nhập.
> Chỉ được encode thành số lúc tính cosine ở Stage 2, không phải ở đây.


In [ ]:
SEMANTIC = ['beach', 'history', 'food', 'nature', 'adventure', 'culture', 'relax', 'photo']
FACTUAL  = ['budget', 'family', 'access', 'crowd']

dataset = []
for place in tfidf:
    row = {
        'place':    place,
        'province': tfidf[place]['province'],
        'lat':      None,  # ← float(coords[place]['lat'])
        'lng':      None,  # ← float(coords[place]['lng'])
    }
    # TODO: thêm 8 semantic features từ tfidf (convert sang float)
    for f in SEMANTIC:
        row[f] = None  # ← float(tfidf[place][f])

    # TODO: thêm 4 factual features từ factual (convert sang float)
    for f in FACTUAL:
        row[f] = None  # ← float(factual[place][f])

    # TODO: thêm best_months — GIỮ NGUYÊN STRING, không convert
    row['best_months'] = None  # ← factual[place]['best_months']

    dataset.append(row)

print(f'Dataset: {len(dataset)} destinations')
print(f'Sample: {list(dataset[0].keys())}')


In [ ]:
# Save
headers = ['place','province','lat','lng'] + SEMANTIC + FACTUAL + ['best_months']
with DATASET_CSV.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=headers)
    writer.writeheader()
    writer.writerows(dataset)
with DATASET_JSON.open('w', encoding='utf-8') as f:
    json.dump(dataset, f, ensure_ascii=False, indent=2)
print(f'Saved: {DATASET_CSV.name}')
print(f'Saved: {DATASET_JSON.name}')


Saved: stage1_dataset.csv
Saved: stage1_dataset.json


In [ ]:
# Score range check — tất cả phải trong [0, 1]
print('Feature score ranges:')
for feat in SEMANTIC + FACTUAL:
    vals = [row[feat] for row in dataset]
    print(f'  {feat:<12} min={min(vals):.3f}  max={max(vals):.3f}')
